# Phase 3e — Approach 5: Vision Transformer (ViT-B/16)

**CSC3109 Machine Learning | Group 30**

## What Is a Vision Transformer?

Vision Transformers (ViT, 2020) are fundamentally different from CNNs. Instead of sliding filters across the image, ViT:

1. **Splits the image into patches** (16×16 pixels each → 196 patches for a 224×224 image)
2. **Embeds each patch** as a vector (like a word in NLP)
3. **Applies self-attention** — each patch can attend to every other patch
4. **Classifies** based on a special `[CLS]` token that aggregates global information

```
224×224 image → split into 196 patches of 16×16
   ↓
Each patch → linear embedding (768-dimensional vector)
   ↓
Add positional encoding (tell the model where each patch is)
   ↓
12 Transformer encoder blocks (multi-head self-attention)
   ↓
CLS token → Linear(768 → 4)
   ↓
4 class scores
```

## Why ViT for Aerial Imagery?

Aerial images often have important **global context** — e.g., the relationship between a central structure and surrounding patterns. CNNs are inherently local (small filter windows). Attention is inherently global — any patch can directly attend to any other patch, regardless of distance.

This makes ViT particularly promising for distinguishing, say, circular wastewater tanks from circular oil-well pads — the surrounding environment matters.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import torch
import torch.nn as nn
import torch.optim as optim
import timm  # library with hundreds of pre-trained vision models

from dataset import get_data_loaders
from train_utils import train_model, plot_training_curves, plot_confusion_matrix, \
                        get_all_predictions, print_classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPORT_DIR = REPO_ROOT / 'report'
print(f'Device: {DEVICE}')
print(f'timm version: {timm.__version__}')

In [ ]:
train_loader, val_loader, cls2idx, idx2cls = get_data_loaders(
    data_dir=REPO_ROOT / 'data', batch_size=16, num_workers=0
    # batch_size=16 (smaller) because ViT uses more memory
)
NUM_CLASSES = len(cls2idx)

---
## Step 2 — Load ViT-B/16

`vit_base_patch16_224` means:
- `base` — medium-sized ViT (~86M parameters)
- `patch16` — each patch is 16×16 pixels
- `224` — expects 224×224 input images

In [ ]:
# Create ViT-B/16 with ImageNet-21k pre-trained weights
# num_classes=4 replaces the original head automatically
model = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True,
    num_classes=NUM_CLASSES
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'ViT-B/16 loaded.')
print(f'Trainable parameters: {trainable:,}')
print('Strategy: full fine-tuning (all transformer blocks unlocked)')
print('Note: ViT is larger than ResNet-50 — training will be slower on CPU')
print('Recommendation: run on Google Colab with GPU for this approach')

---
## Step 3 — Configure Training

ViT benefits from a **warmup** schedule — start with a very low learning rate and gradually increase it before decaying. This prevents instability in the early epochs when the random head interacts with the pre-trained backbone.

We use a simple CosineAnnealing here. For best results on a GPU, use a warmup + cosine schedule (e.g. via `timm.scheduler`).

In [ ]:
NUM_EPOCHS = 30

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.05  # ViT benefits from stronger weight decay
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-7
)

print(f'Optimiser: AdamW (lr=1e-4, weight_decay=0.05)')
print(f'Scheduler: CosineAnnealing over {NUM_EPOCHS} epochs')
print(f'Batch size: 16 (reduced to fit ViT memory requirements)')

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    save_name='vit_b16',
    device=DEVICE
)

In [ ]:
plot_training_curves(
    history,
    title='Approach 5 — Vision Transformer (ViT-B/16)',
    save_path=REPORT_DIR / 'approach5_vit_curves.png'
)

In [ ]:
all_labels, all_preds = get_all_predictions(model, val_loader, DEVICE)
print('=== Approach 5: ViT-B/16 — Validation Results ===')
print_classification_report(all_labels, all_preds)
print(f'Overall Accuracy: {(all_labels == all_preds).mean():.4f}')

In [ ]:
plot_confusion_matrix(
    all_labels, all_preds,
    title='Approach 5 — Vision Transformer (ViT-B/16)',
    save_path=REPORT_DIR / 'approach5_vit_confusion.png'
)

---
## Observations for Report

**Expected outcome:** ViT may outperform CNNs if global context is important for distinguishing these categories. However, ViT typically needs more data to fine-tune well — on only 600 training images/class it may show higher variance.

**Strengths:**
- Captures global spatial relationships between patches
- State-of-the-art direction in computer vision
- Pre-trained on ImageNet-21k (much larger than ImageNet-1k)

**Weaknesses:**
- Large model (86M params) — more prone to overfitting on small datasets
- Computationally expensive on CPU
- Less inductive bias than CNN (CNNs naturally understand translation invariance; ViT must learn it from data)

Best model saved to `models/vit_b16_best.pth`

**Next:** `04_hyperparameter_tuning.ipynb` — tune the best-performing approach further